# SecureCode AI — Full-Scale Classical Baselines (Devign Benchmark)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0Tusharbhardwaj/nlp/blob/main/notebooks/02_full_scale_baselines.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-0Tusharbhardwaj%2Fnlp-blue.svg)](https://github.com/0Tusharbhardwaj/nlp)

## 1. Scientific Objective
Train and evaluate classical machine learning baselines on the **complete official CodeXGLUE / Devign benchmark**:
- **Train Split:** 21,854 C functions
- **Validation Split:** 2,732 C functions (used exclusively for hyperparameter tuning & threshold selection)
- **Test Split:** 2,732 C functions (evaluated once at the very end on untouched data)

### Models Evaluated:
1. **TF-IDF + Logistic Regression (L2 regularized, class-balanced)**
2. **TF-IDF + Linear Support Vector Machine (LinearSVC with Platt probability calibration)**

### Target Metrics:
Accuracy, Precision, Recall, Macro F1, Binary F1, False Negative Rate (FNR = 1 - Recall), ROC-AUC, PR-AUC, Confusion Matrix.

### Step 1: Environment Setup & Google Drive Mounting
Mounts Google Drive to persist trained models, evaluation metrics, and figures across Colab runtimes.

In [ ]:
import os
import sys
from pathlib import Path

# Check if running inside Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    output_dir = Path('/content/drive/MyDrive/SecureCode-AI/baselines')
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f'[OK] Google Drive mounted. Artifacts will save to: {output_dir}')
except ImportError:
    output_dir = Path('results/baselines')
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f'[OK] Running locally. Artifacts will save to: {output_dir}')

### Step 2: Clone Repository & Install Reproducible Dependencies

In [ ]:
import subprocess
from pathlib import Path

if Path('nlp').exists():
    %cd nlp
    !git pull origin main
elif Path('datasets').exists():
    !git pull origin main
else:
    !git clone https://github.com/0Tusharbhardwaj/nlp.git
    %cd nlp

!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib pyarrow pyyaml
print('[OK] Environment dependencies installed successfully.')

### Step 3: Load Full-Scale CodeXGLUE / Devign Partitions
Loads the 27,318 official samples directly from parquet or official Hugging Face mirror.

In [ ]:
import json
import pandas as pd
from pathlib import Path

train_pq = Path('datasets/raw/full/devign/train.parquet')
if not train_pq.exists():
    print('[!] Full datasets not detected locally. Running download pipeline...')
    !python scripts/download_full_datasets.py

train_df = pd.read_parquet('datasets/raw/full/devign/train.parquet')
val_df = pd.read_parquet('datasets/raw/full/devign/validation.parquet')
test_df = pd.read_parquet('datasets/raw/full/devign/test.parquet')

print(f'Devign Train:      {len(train_df):,} samples | Vuln: {train_df["target"].sum():,} ({train_df["target"].mean():.2%})')
print(f'Devign Validation: {len(val_df):,} samples | Vuln: {val_df["target"].sum():,} ({val_df["target"].mean():.2%})')
print(f'Devign Test:       {len(test_df):,} samples | Vuln: {test_df["target"].sum():,} ({test_df["target"].mean():.2%})')
print(f'Total Samples:     {len(train_df) + len(val_df) + len(test_df):,} samples')

X_train = train_df['func'].tolist()
y_train = train_df['target'].astype(int).tolist()

X_val = val_df['func'].tolist()
y_val = val_df['target'].astype(int).tolist()

X_test = test_df['func'].tolist()
y_test = test_df['target'].astype(int).tolist()

### Step 4: Subword & Word-Level TF-IDF Feature Extraction
Fits a scalable TF-IDF vectorizer strictly on `X_train` to prevent data leakage.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import time

print('Fitting TF-IDF Vectorizer on 21,854 training functions...')
t0 = time.time()

vectorizer = TfidfVectorizer(
    token_pattern=r'\b\w+\b',
    ngram_range=(1, 2),
    max_features=50000,
    min_df=3,
    sublinear_tf=True,
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print(f'[OK] Vectorization complete in {time.time() - t0:.2f}s')
print(f'Vocabulary Size: {len(vectorizer.vocabulary_):,} features')
print(f'Train matrix shape: {X_train_vec.shape}')
print(f'Val matrix shape:   {X_val_vec.shape}')
print(f'Test matrix shape:  {X_test_vec.shape}')

### Step 5: Model 1 — Logistic Regression (Tuned on Validation Set)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

best_c = 1.0
best_val_f1 = 0.0
candidate_c = [0.1, 0.5, 1.0, 2.0, 5.0]

print('Tuning Logistic Regression C parameter on Validation Split...')
for c in candidate_c:
    clf = LogisticRegression(C=c, max_iter=1000, class_weight='balanced', random_state=42)
    clf.fit(X_train_vec, y_train)
    preds = clf.predict(X_val_vec)
    f1 = f1_score(y_val, preds, average='macro')
    print(f'  C={c:4.1f} -> Validation Macro F1 = {f1:.4f}')
    if f1 > best_val_f1:
        best_val_f1 = f1
        best_c = c

print(f'\n[OK] Optimal C: {best_c} (Validation Macro F1: {best_val_f1:.4f})')
lr_model = LogisticRegression(C=best_c, max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train_vec, y_train)

### Step 6: Model 2 — Linear SVM (Calibrated on Validation Set)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

print('Training Calibrated Linear SVM...')
t0 = time.time()
base_svm = LinearSVC(C=1.0, max_iter=2000, class_weight='balanced', random_state=42)
svm_model = CalibratedClassifierCV(base_svm, cv=3)
svm_model.fit(X_train_vec, y_train)
val_preds = svm_model.predict(X_val_vec)
val_f1 = f1_score(y_val, val_preds, average='macro')
print(f'[OK] Linear SVM trained in {time.time() - t0:.2f}s | Validation Macro F1 = {val_f1:.4f}')

### Step 7: Final Untouched Test Set Evaluation & Metrics Computation

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
import numpy as np

def evaluate_model(name, model, X_mat, y_true):
    preds = model.predict(X_mat)
    probs = model.predict_proba(X_mat)[:, 1]
    cm = confusion_matrix(y_true, preds)
    tn, fp, fn, tp = cm.ravel()
    
    acc = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    macro_f1 = f1_score(y_true, preds, average='macro')
    binary_f1 = f1_score(y_true, preds, average='binary')
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    roc_auc = roc_auc_score(y_true, probs)
    
    print(f'================== {name} (Devign Test Split) ==================')
    print(f'Accuracy:        {acc:.4f}')
    print(f'Precision:       {prec:.4f}')
    print(f'Recall:          {rec:.4f}')
    print(f'Macro F1:        {macro_f1:.4f}')
    print(f'Binary F1:       {binary_f1:.4f}')
    print(f'False Neg Rate:  {fnr:.4f} (Missed {fn:,} of {fn+tp:,} vulnerabilities)')
    print(f'ROC-AUC:         {roc_auc:.4f}')
    print(f'Confusion Matrix:\n{cm}')
    print('=================================================================\n')
    
    return {
        'accuracy': float(acc),
        'precision': float(prec),
        'recall': float(rec),
        'macro_f1': float(macro_f1),
        'binary_f1': float(binary_f1),
        'false_negative_rate': float(fnr),
        'roc_auc': float(roc_auc),
        'confusion_matrix': cm.tolist(),
    }

lr_results = evaluate_model('TF-IDF Logistic Regression', lr_model, X_test_vec, y_test)
svm_results = evaluate_model('TF-IDF Linear SVM (Calibrated)', svm_model, X_test_vec, y_test)

### Step 8: Export Models, Confusion Matrix Figures, and Metrics JSON

In [ ]:
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Save models and vectorizer to Drive or local output
joblib.dump(vectorizer, output_dir / 'tfidf_vectorizer_full.joblib')
joblib.dump(lr_model, output_dir / 'logistic_regression_full.joblib')
joblib.dump(svm_model, output_dir / 'linear_svm_full.joblib')
print(f'[OK] Models saved to: {output_dir}')

# Plot and save confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(lr_results['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Safe', 'Vuln'], yticklabels=['Safe', 'Vuln'])
axes[0].set_title(f'TF-IDF Logistic Regression\n(Macro F1: {lr_results["macro_f1"]:.3f})')
axes[0].set_ylabel('Ground Truth')
axes[0].set_xlabel('Predicted')

sns.heatmap(svm_results['confusion_matrix'], annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Safe', 'Vuln'], yticklabels=['Safe', 'Vuln'])
axes[1].set_title(f'TF-IDF Linear SVM\n(Macro F1: {svm_results["macro_f1"]:.3f})')
axes[1].set_ylabel('Ground Truth')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
fig_path = output_dir / 'baselines_devign_test_cm.png'
plt.savefig(fig_path, dpi=300)
plt.show()
print(f'[OK] Confusion matrix figure saved to: {fig_path}')

# Save metrics JSON
metrics_path = output_dir / 'full_scale_baselines_metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump({
        'dataset': 'CodeXGLUE / Devign (Official 27,318 samples)',
        'train_samples': len(X_train),
        'val_samples': len(X_val),
        'test_samples': len(X_test),
        'results': {
            'logistic_regression': lr_results,
            'linear_svm': svm_results,
        }
    }, f, indent=2)
print(f'[OK] Metrics JSON saved to: {metrics_path}')